In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [13]:


# 1. Load the dataset
# Assuming 'Metro_Interstate_Traffic_Volume.csv'
df = pd.read_csv('../data/Metro_Interstate_Traffic_Volume.csv')


print(df.isna().sum())

holiday                48143
temp                       0
rain_1h                    0
snow_1h                    0
clouds_all                 0
weather_main               0
weather_description        0
date_time                  0
traffic_volume             0
dtype: int64


In [14]:
print(df.head())

  holiday    temp  rain_1h  snow_1h  clouds_all weather_main  \
0     NaN  288.28      0.0      0.0          40       Clouds   
1     NaN  289.36      0.0      0.0          75       Clouds   
2     NaN  289.58      0.0      0.0          90       Clouds   
3     NaN  290.13      0.0      0.0          90       Clouds   
4     NaN  291.14      0.0      0.0          75       Clouds   

  weather_description            date_time  traffic_volume  
0    scattered clouds  2012-10-02 09:00:00            5545  
1       broken clouds  2012-10-02 10:00:00            4516  
2     overcast clouds  2012-10-02 11:00:00            4767  
3     overcast clouds  2012-10-02 12:00:00            5026  
4       broken clouds  2012-10-02 13:00:00            4918  


In [15]:

# 2. Extract Time Features from 'date_time'
df['date_time'] = pd.to_datetime(df['date_time'])
df['hour'] = df['date_time'].dt.hour
df['day_of_week'] = df['date_time'].dt.dayofweek
df['month'] = df['date_time'].dt.month

# 3. Cyclical Encoding for Time (Gold Standard for KNN)
def encode_cyclical(data, col, max_val):
    data[col + '_sin'] = np.sin(2 * np.pi * data[col] / max_val)
    data[col + '_cos'] = np.cos(2 * np.pi * data[col] / max_val)
    return data.drop(columns=[col])

df = encode_cyclical(df, 'hour', 24)
df = encode_cyclical(df, 'day_of_week', 7)
df = encode_cyclical(df, 'month', 12)

# 4. Handle Categorical Features (One-Hot Encoding)
# We drop 'weather_description' to avoid exploding the feature space.
# We also drop the original 'date_time' as it's now encoded.
df = df.drop(columns=['date_time', 'weather_description', 'holiday'])

# Convert 'holiday' and 'weather_main' into binary dummy variables (0 or 1)
df = pd.get_dummies(df, columns=['weather_main'], drop_first=False)

# 5. Define Features and Target
X = df.drop(columns=['traffic_volume'])
y = df['traffic_volume']

# 6. Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7. Min-Max Scaling (Parity for KNN)
# Continuous features need to be squashed to [0, 1] to match the dummy variables.
# The cyclical features are [-1, 1], so scaling them to [0, 1] ensures 100% parity.
scaler = StandardScaler()

# We can safely scale the entire dataframe because the dummy variables are 0/1.
# Min-Max scaling a 0/1 column leaves it exactly as 0/1!
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), 
    columns=X_train.columns
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), 
    columns=X_test.columns
)

# 8. Reconstruct and Save
train_final = X_train_scaled.copy()
train_final['y'] = y_train.values

test_final = X_test_scaled.copy()
test_final['y'] = y_test.values

train_final.to_csv('../data/traffic_preprocessed_train.csv', index=False)
test_final.to_csv('../data/traffic_preprocessed_test.csv', index=False)

print("--- Preprocessing Complete ---")
print(f"Total features after One-Hot Encoding: {len(X.columns)}")
print(f"Target: traffic_volume")

--- Preprocessing Complete ---
Total features after One-Hot Encoding: 21
Target: traffic_volume
